# HydraY NNUE - strato nascosto a 1024, sui dati con i finali

Runtime → Cambia tipo di runtime → **GPU (T4)**, poi Runtime → **Esegui tutte**.
Durata ~1h30. **Non lasciare la scheda inattiva.**

### A cosa serve
Raddoppia lo strato nascosto, da 512 a 1024, tenendo **tutto il resto identico**
alla rete appena spedita: stesso dataset (1.283.450.302 posizioni, bucket 0 al
7,1%), stessi 4 king bucket specchiati, stessi 8 bucket di uscita, stesso
schedule da 40 superbatch, stesso validation set. L'unica variabile e' la
**capacita'**.

### Perche' rifarlo, visto che la 1024 ha gia' perso
Era il 3 agosto, e girava sui dati **senza finali**. In quel dataset il bucket 0
valeva lo 0,23%: la rete non aveva niente da imparare in quella regione, e la
capacita' in piu' non aveva dove andare a finire.

Da allora sono cambiate due cose che riaprono la domanda:

1. il bucket 0 e' passato dallo 0,23% al **7,1%** (due batch di finali, +27,9 e
   +7,4 Elo). C'e' materiale nuovo che una rete piu' grande puo' assorbire;
2. A5 aveva misurato che piu' dati su una 512 non danno niente (−7,30 ±8,61).
   Restava aperto se il tetto fosse **i dati** o **la capacita'** - e si
   distingue solo cambiando la capacita'.

Due risposte al prezzo di una: se vince, hai la rete nuova; se vince di molto,
il tetto era la capacita' e la questione dati si riapre tutta.

### Il prezzo, che e' gia' dentro il verdetto
Il binario paghera' **15-20% di NPS**: l'accumulatore e' il 13,6% dei cicli e il
forward il 7,5%, ed entrambi raddoppiano. Lo SPRT gira sotto controllo di tempo,
quindi quel costo e' gia' scontato nel risultato - **non va sottratto a mano**.
In pratica la 1024 deve valere piu' di ~15 Elo di sola valutazione per uscire
in pari.

### Perche' NON e' a tappe
Il dataset scompattato e' 38,3 GiB e sta sul disco del runtime: si addestra in
un colpo solo, come ha fatto la 512. Niente `STAGE_END`, niente fette, niente
tappe da eseguire in ordine.


In [ ]:
# --- helper: qualunque comando fallito ferma il notebook, e l'output si vede ---
import subprocess, os, sys, json

def sh(cmd):
    # L'output va riletto e ristampato da Python: subprocess.run() senza capture
    # scrive sui file descriptor del KERNEL, che Colab non mostra nella cella.
    print('$', cmd, flush=True)
    p = subprocess.Popen(cmd, shell=True, executable='/bin/bash',
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    if p.wait() != 0:
        raise RuntimeError(f'FALLITO (exit {p.returncode}): {cmd}')

sh('nvidia-smi --query-gpu=name,memory.total --format=csv')
sh('df -h /content | tail -1')
print('\nGPU presente. Se la riga sopra non mostra una T4, cambia runtime.')


In [ ]:
# --- Drive + dataset (le STESSE due parti usate dalla rete a 512) ---
from google.colab import drive
drive.mount('/content/drive')

import glob
def find(name):
    hits = glob.glob(f'/content/drive/MyDrive/**/{name}', recursive=True)
    assert hits, f'{name} non trovato su Drive'
    return hits[0]
P1, P2 = find('hydray_v6_part1.bin.zst'), find('hydray_v6_part2.bin.zst')
os.environ['P1'], os.environ['P2'] = P1, P2
print('parte 1:', P1, os.path.getsize(P1), 'byte')
print('parte 2:', P2, os.path.getsize(P2), 'byte')

NET_ID   = 'hydray-1024-eg103m'
TOTAL_SB = 40
TRAINER  = '/content/th/nnue/trainer'


In [ ]:
# --- Rust ---
sh("curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal")
sh('$HOME/.cargo/bin/cargo --version')


In [ ]:
# --- clone + verifica architettura ---
# La verifica non e' cerimoniale: un run precedente ha addestrato per un'ora
# un'architettura diversa da quella creduta, perche' il clone era sbagliato e il
# nome del checkpoint non dice nulla sul contenuto.
sh('rm -rf /content/th')
sh('git clone --depth 1 --branch nnue-1024 https://github.com/ThomasGhione/HydraY /content/th')

src = open(f'{TRAINER}/src/bin/sanity.rs').read()
assert 'const HIDDEN: usize = 1024;' in src, 'NON e il branch a 1024 neuroni'
assert 'const INPUT_BUCKETS: usize = 4;' in src, 'i king bucket devono restare 4'
tr = open(f'{TRAINER}/src/bin/trainer.rs').read()
assert 'const HIDDEN_SIZE: usize = 1024;' in tr, 'trainer.rs non e a 1024'
print('branch nnue-1024, 1024 neuroni, 4 king bucket: ok')


In [ ]:
# --- decompressione in due parti (ogni zstd esce e libera la cache di Drive) ---
sh('apt-get -qq install -y zstd >/dev/null')

TOT, HALF = 41070409664, 20535204832
st = os.statvfs('/content'); free_gb = st.f_bavail * st.f_frsize / 2**30
print(f'liberi {free_gb:.1f} GiB, picco atteso ~50 GiB')
assert free_gb > 52, 'disco insufficiente'

sh('zstd -d -T0 --long=27 -c "$P1" > /content/data.bin')
assert os.path.getsize('/content/data.bin') == HALF, 'parte 1 di taglia inattesa'
print('parte 1 ok'); sh('df -h /content | tail -1')

sh('zstd -d -T0 --long=27 -c "$P2" >> /content/data.bin')
SIZE = os.path.getsize('/content/data.bin')
print('data.bin:', SIZE, 'byte =', SIZE // 32, 'posizioni')
assert SIZE == TOT, f'taglia finale inattesa: {SIZE}'
sh('df -h /content | tail -1')


In [ ]:
# --- validation set: 1000 MiB dalla coda ---
# ATTENZIONE: questa fetta resta anche dentro data.bin, quindi la "validation"
# loss e' contaminata. Vale SOLO come confronto relativo con la 512, che e'
# stata addestrata esattamente allo stesso modo sullo stesso file: non e' una
# stima di generalizzazione. Il giudice resta lo SPRT.
TEST_MIB = 1000
skip_mib = SIZE // (1024*1024) - TEST_MIB
sh(f'dd if=/content/data.bin bs=1M skip={skip_mib} count={TEST_MIB} of=/content/test.bin status=progress')
print('test.bin:', os.path.getsize('/content/test.bin'), 'byte')


In [ ]:
# --- training: un solo run, 40 superbatch (identico alla 512) ---
sh(f'cd {TRAINER} && PATH=$HOME/.cargo/bin:$PATH CUDA_PATH=/usr/local/cuda '
   f'TEST_PATH=/content/test.bin '
   f'cargo run -r --bin trainer --features cuda -- '
   f'/content/data.bin {TOTAL_SB} {NET_ID}')


In [ ]:
# --- verifica finale e salvataggio su Drive ---
final = f'{TRAINER}/checkpoints/{NET_ID}-{TOTAL_SB}/quantised.bin'
sz = os.path.getsize(final)
# payload 6.326.288 + padding a 64 byte. La rete a 512 pesa 3.163.200: la
# taglia e' il controllo piu' rapido che l'architettura sia quella giusta.
assert 6326288 <= sz < 6326288 + 64, f'taglia {sz}: NON e la rete a 1024'
print('quantised.bin:', sz, 'byte - 1024 neuroni confermati\n')

sh(f'cd {TRAINER} && PATH=$HOME/.cargo/bin:$PATH cargo run -r --bin sanity -- {final}')
sh(f'cp -r {TRAINER}/checkpoints/{NET_ID}-{TOTAL_SB} /content/drive/MyDrive/')
print('\n' + '='*64)
print('CONFRONTO con la rete a 512 addestrata sugli STESSI dati:')
print('  startpos      45      middlegame   933')
print('  KQvK         662      KRPvKR        70')
print('  re attivi    103      loss finale   0.013259')
print('Riporta: loss di training, VALIDATION loss, e i sanity qui sopra.')
print('='*64)


## Come leggere il risultato

Il confronto e' **testa a testa** contro la rete a 512 addestrata sullo stesso
identico dataset con lo stesso schedule. L'unica variabile e' `HIDDEN`, quindi
il risultato e' il valore della capacita' in purezza - non una sottrazione fra
due misure indipendenti, che sommerebbe gli errori.

Prima dello SPRT, i due controlli da fare in un minuto:

- **taglia del file**: 6.326.336 byte. Se ne legge 3.163.200 hai addestrato una
  512 e il clone era sbagliato;
- **i sanity eval**, contro i valori della 512 stampati dalla cella sopra. Se
  KQvK e mediogioco crollano, la rete e' peggiore e lo sai subito.

Poi lo SPRT, che e' l'unico giudice: il costo di NPS della rete piu' grande
e' gia' dentro quel numero.

- **vince** → il tetto era la capacita', non i dati. Si riapre anche la
  questione dei dati che A5 sembrava aver chiuso, e diventa sensato tornare a
  generare (e a rifare il test 8k vs 12k nodi per mossa)
- **pareggia o perde** → la 512 non e' limitata dalla larghezza. La strada
  seguente non e' piu' larga ma piu' **profonda**: un layer intermedio
  (512x2 → 32 → 1), che e' un progetto C++ vero perche' richiede la
  riquantizzazione a int8 del forward
